# Stage 9 — Debit spread scale-up (218 stocks + 8 indices, 1H)

3-stock validation (BHARTIARTL, RELIANCE, NIFTY) showed 2/3 with a real edge
and 1 near-breakeven — encouraging but too thin a sample to trust. Round 1
of the full-universe run (214 instruments, 2024-11-15 to now, ~21 months)
gave 59.3% of instruments a positive avg R, but per-instrument trade counts
were thin (30-42 for the top names) since this signal only fires ~1-2x/month
per stock.

**This run extends the window to 2023-01-01 (~31 months)** for a larger
per-instrument sample — a moderate extension, not the full ~4yr window used
for the Supertrend strategies, since this is a short-term momentum signal
and reaching too far back risks mixing in an older, less-representative
volatility regime (see README assumptions log for the full reasoning).
force_refresh=True this run, since the window changed for every instrument.

New: `src/debit_spread_scale_pipeline.py` — parallel to `scale_pipeline.py` +
`compare_stocks.py`, not a modification of either. Same discipline: resume
support, isolated failure handling, rate-limit delay.

Uses the warm-up-buffered fetch window (a ~6-week buffer before the
intended analysis start date) so early
EMA/ADX instability doesn't contaminate signals.

This will take longer than round 1 (~31 months of 1H data vs ~21 — more API
chunks per instrument) — resume support means it's safe to re-run if
interrupted partway.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

213

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/213] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6195 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 75 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/360ONE_trades.csv
[1/213] 360ONE — done (trades=75, win_rate=50.7, avg_r=0.19)
[2/213] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6195 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 59 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,75.0,75.0,0.0,50.7,0.19,0.15,"{'STOP': 31, 'TARGET': 27, 'TIME': 17}",7.0,360ONE,SUCCESS,
1,59.0,59.0,0.0,45.8,0.08,-1.00,"{'STOP': 31, 'TARGET': 22, 'TIME': 6}",6.0,ABB,SUCCESS,
2,72.0,72.0,0.0,37.5,-0.15,-1.00,"{'STOP': 42, 'TARGET': 20, 'TIME': 10}",6.0,ABCAPITAL,SUCCESS,
3,68.0,68.0,0.0,52.9,0.29,0.28,"{'TARGET': 30, 'STOP': 28, 'TIME': 10}",5.0,ADANIENSOL,SUCCESS,
4,66.0,66.0,0.0,39.4,-0.10,-1.00,"{'STOP': 37, 'TARGET': 18, 'TIME': 11}",6.0,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...
208,58.0,58.0,0.0,48.3,0.09,-0.67,"{'STOP': 29, 'TARGET': 21, 'TIME': 8}",7.0,VOLTAS,SUCCESS,
209,32.0,32.0,0.0,53.1,0.19,0.04,"{'STOP': 14, 'TARGET': 13, 'TIME': 5}",5.5,WAAREEENER,SUCCESS,
210,71.0,70.0,1.0,51.4,0.21,0.18,"{'STOP': 32, 'TARGET': 30, 'TIME': 8}",6.0,WIPRO,SUCCESS,
211,83.0,83.0,0.0,32.5,-0.19,-1.00,"{'STOP': 54, 'TARGET': 25, 'TIME': 4}",5.0,YESBANK,SUCCESS,


## Indices — same 6 as the rest of this project (Stage 5), separate token lookup

In [3]:
from src.debit_spread_scale_pipeline import process_instrument

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

import pandas as pd
index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6199 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 60 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6199 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 58 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,60,60,0,51.7,0.27,0.22,"{'TARGET': 27, 'STOP': 26, 'TIME': 7}",5.5,NIFTY,SUCCESS,
1,58,58,0,48.3,0.17,-0.32,"{'STOP': 26, 'TARGET': 23, 'TIME': 9}",6.0,BANKNIFTY,SUCCESS,
2,50,50,0,50.0,0.19,0.08,"{'STOP': 23, 'TARGET': 19, 'TIME': 8}",6.0,SENSEX,SUCCESS,
3,67,67,0,47.8,0.13,-1.00,"{'STOP': 34, 'TARGET': 26, 'TIME': 7}",6.0,NIFTYNXT50,SUCCESS,
4,84,84,0,48.8,0.17,-0.44,"{'STOP': 41, 'TARGET': 34, 'TIME': 9}",5.0,MIDCPNIFTY,SUCCESS,
5,73,73,0,45.2,0.06,-1.00,"{'STOP': 37, 'TARGET': 27, 'TIME': 9}",6.0,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved debit spread comparison of 214 instruments to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/DEBIT_SPREAD_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,SWIGGY,29.0,29.0,65.5,0.54,1.50,6.0
1,POWERINDIA,67.0,67.0,64.2,0.54,1.50,5.0
2,CDSL,68.0,68.0,57.4,0.41,1.46,6.0
3,HAL,69.0,68.0,55.9,0.37,1.06,6.5
4,PFC,61.0,61.0,54.1,0.36,0.95,6.0
...,...,...,...,...,...,...,...
209,YESBANK,83.0,83.0,32.5,-0.19,-1.00,5.0
210,MPHASIS,83.0,83.0,34.9,-0.22,-1.00,6.0
211,MARICO,76.0,76.0,35.5,-0.23,-1.00,7.0
212,BRITANNIA,73.0,72.0,31.9,-0.25,-1.00,6.0


In [5]:
# The actual answer to "does signal quality hold up": across every instrument
# with enough trades to mean anything (>= 15 closed trades, matching the
# minimum this project already uses in reversion_test.py), what fraction
# show a positive avg_r_achieved, and what does the distribution look like?
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())

Instruments processed: 214
Instruments with >= 15 closed trades: 214
Fraction with positive avg_r_achieved: 69.2%

avg_r_achieved distribution:
count    214.000000
mean       0.076963
std        0.139734
min       -0.270000
25%       -0.020000
50%        0.080000
75%        0.170000
max        0.540000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    214.000000
mean      45.329439
std        5.755864
min       31.900000
25%       41.300000
50%       45.300000
75%       49.200000
max       65.500000
Name: win_rate_pct, dtype: float64
